# 🤖 ТАРС v4 — Автономное обучение

**Всё автономно. Всё сохраняется на Google Drive. Переподключение = продолжение.**

| Что | Где |
|---|---|
| Датасеты | `MyDrive/TarsData/` |
| Модели + чекпоинты | `MyDrive/TarsModels/` |
| LEANN память | `MyDrive/TarsMemory/` |
| Логи | `MyDrive/TarsModels/mega_train.log` |

### Инструкция
1. **Runtime → Change runtime type → L4 GPU** (или T4 бесплатно)
2. **Запустить ячейку 1** — всё остальное автоматически
3. Если отключилось — просто **запустить ячейку 1 заново** (всё продолжится с места остановки)

In [ ]:
#@title 🚀 **ЗАПУСТИ МЕНЯ** — полностью автономное обучение { display-mode: "form" }
# ============================================================
# Установка, скачивание, обучение, сохранение — всё в одной ячейке
# ============================================================

import os, sys, time, shutil, subprocess, signal
from pathlib import Path

# ==== 1. GOOGLE DRIVE ====
print('═' * 60)
print('  🤖 ТАРС v4 — Автономное обучение')
print('═' * 60)
print()

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DATA   = Path('/content/drive/MyDrive/TarsData')
DRIVE_MODELS = Path('/content/drive/MyDrive/TarsModels')
DRIVE_MEMORY = Path('/content/drive/MyDrive/TarsMemory')
DRIVE_DATA.mkdir(parents=True, exist_ok=True)
DRIVE_MODELS.mkdir(parents=True, exist_ok=True)
DRIVE_MEMORY.mkdir(parents=True, exist_ok=True)

# Лог на Drive (не теряется)
DRIVE_LOG = DRIVE_MODELS / 'mega_train.log'
print(f'  ☁️  Drive: data={DRIVE_DATA}')
print(f'           models={DRIVE_MODELS}')
print(f'           memory={DRIVE_MEMORY}')

# Показать что уже есть
existing_data = list(DRIVE_DATA.glob('*.txt'))
existing_models = list(DRIVE_MODELS.glob('*.pt'))
if existing_data:
    mb = sum(f.stat().st_size for f in existing_data) / 1024**2
    print(f'  📂 Данные: {len(existing_data)} файлов, {mb:.0f} MB')
if existing_models:
    mb = sum(f.stat().st_size for f in existing_models) / 1024**2
    print(f'  🧠 Модели: {len(existing_models)} файлов, {mb:.0f} MB')
print()

# ==== 2. GPU CHECK ====
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print()

# ==== 3. CLONE / UPDATE CODE ====
os.chdir('/content')
if os.path.exists('TarsSSM-Py'):
    # Обновить код без потери данных
    os.chdir('/content/TarsSSM-Py')
    !git pull --rebase 2>/dev/null || true
else:
    !git clone https://github.com/Vazilll/TarsSSM-Py.git
    os.chdir('/content/TarsSSM-Py')

ROOT = Path('/content/TarsSSM-Py')
print(f'  📁 Root: {ROOT}')

# ==== 4. SYMLINKS: data/ → Drive, models/ → Drive ====
def safe_symlink(local, drive_target):
    """Create symlink local -> drive, migrating existing files."""
    if local.is_symlink():
        return  # уже симлинк
    if local.exists():
        # Перенести файлы на Drive
        for f in local.rglob('*'):
            if f.is_file():
                rel = f.relative_to(local)
                dest = drive_target / rel
                dest.parent.mkdir(parents=True, exist_ok=True)
                if not dest.exists():
                    shutil.move(str(f), str(dest))
        shutil.rmtree(str(local))
    local.symlink_to(drive_target)

safe_symlink(ROOT / 'data', DRIVE_DATA)
safe_symlink(ROOT / 'models', DRIVE_MODELS)
print('  🔗 data/ → Drive/TarsData')
print('  🔗 models/ → Drive/TarsModels')
print()

# ==== 5. KEEP-ALIVE (анти-таймаут) ====
# Colab отключает при отсутствии активности.
# Этот код поддерживает соединение во время обучения.
import threading
def _keep_alive():
    """Ping output every 10 min to prevent idle timeout."""
    while True:
        time.sleep(600)
        elapsed = time.time() - _train_start
        h, m = int(elapsed // 3600), int((elapsed % 3600) // 60)
        print(f'  ⏰ Keep-alive: {h}h {m}m — обучение продолжается...')

_train_start = time.time()
_ka = threading.Thread(target=_keep_alive, daemon=True)
_ka.start()

# ==== 6. ГЕНЕРАЦИЯ КОРПУСА ====
# Генерируем корпус личности если его нет
personality_corpus = DRIVE_DATA / 'tars_personality_mega.txt'
if not personality_corpus.exists() or personality_corpus.stat().st_size < 1000:
    print('  📝 Генерация корпуса личности...')
    !cd /content/TarsSSM-Py && python training/generate_tars_corpus.py
    print('  ✅ Корпус сохранён на Drive')
else:
    mb = personality_corpus.stat().st_size / 1024**2
    print(f'  📝 Корпус уже есть: {mb:.1f} MB')
print()

# ==== 7. CHECKPOINT STATE ====
# Определяем откуда продолжать
has_data = len(list(DRIVE_DATA.glob('hf_*.txt'))) > 0 or len(list(DRIVE_DATA.glob('wiki_*.txt'))) > 0
has_model = (DRIVE_MODELS / 'tars_v3' / 'mamba2.pt').exists()

extra_args = []
if has_data:
    extra_args.append('--skip-download')
    print('  ✅ Данные на Drive — пропуск скачивания')
if has_model:
    print('  ✅ Модель на Drive — резюме обучения')
print()

# ==== 8. ОБУЧЕНИЕ ====
print('═' * 60)
print('  🎓 Начало обучения — всё сохраняется на Drive каждые 5 мин')
print('═' * 60)
print()

cmd = [
    sys.executable, 'mega_train.py',
    '--skip-voice',  # голосовые фазы опциональны
    '--drive',       # кеширование на Drive
] + extra_args

# Запуск с логированием в файл и stdout
result = subprocess.run(cmd, cwd=str(ROOT))

# ==== 9. РЕЗУЛЬТАТ ====
elapsed = time.time() - _train_start
hours = elapsed / 3600
minutes = elapsed / 60

print()
print('═' * 60)
if result.returncode == 0:
    print(f'  ✅ ОБУЧЕНИЕ ЗАВЕРШЕНО за {hours:.1f}ч ({minutes:.0f} мин)!')
    print()
    # Показать модели
    for f in sorted(DRIVE_MODELS.rglob('*.pt')):
        mb = f.stat().st_size / 1024**2
        print(f'    💾 {f.name}: {mb:.1f} MB')
    print()
    print('  📁 Данные:  MyDrive/TarsData/')
    print('  🧠 Модели: MyDrive/TarsModels/')
    print('  🚀 Запуск:  python launch_tars.py')
else:
    print(f'  ⚠️  Ошибка (код {result.returncode})')
    print(f'     Время: {minutes:.0f} мин')
    print()
    print('  Модели сохранены на Drive.')
    print('  ⏯️  Просто перезапустите эту ячейку — обучение продолжится.')
    print('  📋 Логи: MyDrive/TarsModels/mega_train.log')
print('═' * 60)

In [ ]:
#@title 📊 Статус обучения { display-mode: "form" }
import os
from pathlib import Path

DRIVE_DATA   = Path('/content/drive/MyDrive/TarsData')
DRIVE_MODELS = Path('/content/drive/MyDrive/TarsModels')
DRIVE_MEMORY = Path('/content/drive/MyDrive/TarsMemory')

print('═' * 50)
print('  📊 Статус обучения')
print('═' * 50)

# Данные
if DRIVE_DATA.exists():
    txts = list(DRIVE_DATA.glob('*.txt'))
    total = sum(f.stat().st_size for f in txts) / 1024**2
    print(f'  📁 Данные: {len(txts)} файлов, {total:.0f} MB')

# Модели
if DRIVE_MODELS.exists():
    for f in sorted(DRIVE_MODELS.rglob('*.pt')):
        mb = f.stat().st_size / 1024**2
        print(f'  🧠 {f.relative_to(DRIVE_MODELS)}: {mb:.1f} MB')

# LEANN
if DRIVE_MEMORY.exists():
    npz = list(DRIVE_MEMORY.glob('*.npz'))
    if npz:
        mb = sum(f.stat().st_size for f in npz) / 1024**2
        print(f'  🧠 LEANN: {mb:.1f} MB')

# GPU
try:
    import torch
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f'  🎮 VRAM: {alloc:.1f}/{total:.1f} GB')
except: pass

# Последние логи
print()
print('  📋 Последние строки лога:')
log = Path('/content/TarsSSM-Py/mega_train.log')
if log.exists():
    lines = log.read_text(encoding='utf-8', errors='replace').strip().split('\n')
    for l in lines[-15:]:
        print(f'    {l}')
else:
    print('    (нет лога)')
print('═' * 50)

In [ ]:
#@title 📥 Скачать модели локально { display-mode: "form" }
# Скачивает готовые модели с Drive на локальный компьютер
from google.colab import files
from pathlib import Path
import zipfile, os

DRIVE_MODELS = Path('/content/drive/MyDrive/TarsModels')
tars_v3 = DRIVE_MODELS / 'tars_v3'

if tars_v3.exists():
    zip_path = '/content/tars_v3_models.zip'
    print('📦 Упаковка...')
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in tars_v3.rglob('*'):
            if f.is_file():
                zf.write(f, f.relative_to(DRIVE_MODELS))
                print(f'  + {f.name} ({f.stat().st_size / 1024**2:.1f} MB)')
    
    total_mb = os.path.getsize(zip_path) / 1024**2
    print(f'\n💾 Итого: {total_mb:.0f} MB')
    print('📥 Скачивание...')
    files.download(zip_path)
else:
    print('⚠️  Модели не найдены. Сначала запустите обучение.')

### 📝 Полезные команды
```python
# Обучить только Mamba-2 (фаза 4)
!cd /content/TarsSSM-Py && python mega_train.py --phase 4 --drive

# Обучить только CoT (фаза 12)
!cd /content/TarsSSM-Py && python mega_train.py --phase 12 --drive

# Обучить только DPO (фаза 13)
!cd /content/TarsSSM-Py && python mega_train.py --phase 13 --drive

# Быстрый тест (маленькая модель)
!cd /content/TarsSSM-Py && python mega_train.py --quick --drive

# MAX модель (768M params, 8-24ч)
!cd /content/TarsSSM-Py && python mega_train.py --max --drive

# Посмотреть лог
!tail -30 /content/TarsSSM-Py/mega_train.log

# Очистить всё
!rm -rf /content/drive/MyDrive/TarsData/* /content/drive/MyDrive/TarsModels/*
```